### 1. Connect to the data files

In [ ]:

from google.colab import drive
import os

drive.mount('/content/drive')

# Example: Accessing a file in your Google Drive
# Make sure to replace 'MyDrive/path/to/your/file.txt' with the actual path
train_file_path = 'path/to/project_folder/tweet_train.csv'
test_file_path = 'path/to/project_folder/tweet_test.csv'

### 2. Import basic packages

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns

### 3. Exploratory Data Analysis

In [ ]:
tweet_train = pd.read_csv(train_file_path)
tweet_test = pd.read_csv(test_file_path)

In [ ]:
print(tweet_train.info())
print(tweet_train.head())

In [ ]:
# Calculate the percentage of missing values for each column
missing_percentages = tweet_train.isnull().sum() * 100 / len(tweet_train)

# Print the results
missing_percentages


In [ ]:
#Duplicate ID check

duplicate_ids = tweet_train[tweet_train.duplicated(subset=['id'], keep=False)]
duplicate_ids


#### 3.1 General distribution and text analysis of disaster tweets

In [ ]:

import matplotlib.pyplot as plt
# Count the occurrences of each target value
target_counts = tweet_train['target'].value_counts()

# Create the bar plot
plt.figure(figsize=(8, 6))
sns.countplot(x='target', data=tweet_train)
plt.title('Distribution of Disaster vs. Non-Disaster Tweets')
plt.xlabel('Target (0: Non-Disaster, 1: Disaster)')
plt.ylabel('Count')

# Annotate the bars with the counts
for i, count in enumerate(target_counts):
    plt.text(i, count + 50, str(count), ha='center', va='bottom')  # Adjust the vertical offset (50) as needed

plt.show()


In [ ]:


import matplotlib.pyplot as plt
# Calculate the average text length for disaster and non-disaster tweets
tweet_train['text_length'] = tweet_train['text'].apply(len)
average_text_length_by_target = tweet_train.groupby('target')['text_length'].mean()

# Create the histogram
plt.figure(figsize=(8, 6))
plt.hist(tweet_train[tweet_train['target'] == 0]['text_length'], bins=30, alpha=0.5, label='Non-Disaster')
plt.hist(tweet_train[tweet_train['target'] == 1]['text_length'], bins=30, alpha=0.5, label='Disaster')
plt.xlabel('Text Length')
plt.ylabel('Frequency')
plt.title('Distribution of Text Lengths for Disaster and Non-Disaster Tweets')
plt.legend()
plt.show()

# Print the average text lengths
print("Average text length for non-disaster tweets:", average_text_length_by_target[0])
print("Average text length for disaster tweets:", average_text_length_by_target[1])


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Expanded dictionary to normalize locations
location_map = {
    "USA": "United States",
    "U.S.": "United States",
    "America": "United States",
    "United States of America": "United States",
    "Califnoria, USA": "California",
    "California, USA": "California",
    "NY, USA": "New York",
    "New York, NY": "New York",
    "Los Angeles, CA": "Los Angeles",
    "London, UK": "London",
    "Londan": "London",  # Fixing spelling
    "Ontario, Canada": "Canada",
    "Canada, CA": "Canada",
    "UK": "United Kingdom",
    "England": "United Kingdom",
    "Scotland": "United Kingdom",
    "Wales": "United Kingdom",
    "Britain": "United Kingdom",
    "Great Britain": "United Kingdom",
    "Mumbai, India": "Mumbai",
    "Delhi, India": "Delhi",
    "New Delhi": "Delhi",
    "Bangalore, India": "Bangalore",
    "Chennai, India": "Chennai",
    "India": "India",
    "Everywhere": None,  # Remove general/ambiguous locations
    "Worldwide": None,
}

# Standardize locations
def clean_location(loc):
    if pd.isna(loc):
        return None
    loc = loc.strip()
    return location_map.get(loc, loc)  # Replace if in dictionary, else keep original

tweet_train['cleaned_location'] = tweet_train['location'].apply(clean_location)

# Remove empty locations
tweet_train = tweet_train.dropna(subset=['cleaned_location'])

# Count tweets per location after cleaning
location_counts = tweet_train.groupby(['cleaned_location', 'target']).size().unstack(fill_value=0)

# Sort by total tweet count
location_counts['total'] = location_counts.sum(axis=1)
location_counts = location_counts.sort_values(by='total', ascending=False).head(20)  # Top 20 locations

# Plot updated bar graph
plt.figure(figsize=(12, 6))
location_counts[[0, 1]].plot(kind='bar', stacked=True, color=['blue', 'red'], alpha=0.7)

plt.title("Top 20 Locations by Tweet Count (Disaster vs. Non-Disaster)")
plt.xlabel("Location")
plt.ylabel("Tweet Count")
plt.legend(["Non-Disaster Tweets", "Disaster Tweets"])
plt.xticks(rotation=45, ha='right')
plt.tight_layout()

plt.show()


#### 3.2 Semantic Analysis of Disaster and Non-disaster Tweets

In [ ]:
import nltk
from nltk.sentiment import SentimentIntensityAnalyzer

# Download 'punkt_tab', 'vader_lexicon', and 'stopwords'
nltk.download('punkt_tab')
nltk.download('vader_lexicon')
nltk.download('stopwords') # Download the stopwords dataset

sia = SentimentIntensityAnalyzer()

# Function to analyze sentiment of words
def get_sentiment_scores(word_list):
    sentiment_scores = {word: sia.polarity_scores(word)['compound'] for word in word_list}
    return sentiment_scores

# Assuming you have code that computes word frequencies like this (example):
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from collections import Counter

# ... (your text processing code) ...
stop_words = set(stopwords.words('english'))

def process_text(text):
    tokens = word_tokenize(text.lower())
    filtered_tokens = [w for w in tokens if w.isalnum() and w not in stop_words]
    return filtered_tokens

# Process for disaster tweets
disaster_tweets = tweet_train[tweet_train['target'] == 1]['text']
disaster_words = []
for tweet in disaster_tweets:
    disaster_words.extend(process_text(tweet))
disaster_word_counts = Counter(disaster_words)
top_disaster_words = disaster_word_counts.most_common(20)  # Get top 20 disaster words
disaster_sentiments = get_sentiment_scores([word for word, _ in top_disaster_words])

# Process for non-disaster tweets  # This section was missing
non_disaster_tweets = tweet_train[tweet_train['target'] == 0]['text']
non_disaster_words = []
for tweet in non_disaster_tweets:
    non_disaster_words.extend(process_text(tweet))
non_disaster_word_counts = Counter(non_disaster_words)
top_non_disaster_words = non_disaster_word_counts.most_common(20)  # Get top 20 non-disaster words
non_disaster_sentiments = get_sentiment_scores([word for word, _ in top_non_disaster_words]) # Calculate and assign to non_disaster_sentiments

# Print results
print("Disaster Tweet Words Sentiment:", disaster_sentiments)
print("Non-Disaster Tweet Words Sentiment:", non_disaster_sentiments)

In [ ]:
import matplotlib.pyplot as plt
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from collections import Counter
from nltk.sentiment import SentimentIntensityAnalyzer

# Download necessary resources
nltk.download('stopwords')
nltk.download('vader_lexicon')
nltk.download('punkt')

# Initialize sentiment analyzer
sia = SentimentIntensityAnalyzer()
stop_words = set(stopwords.words('english'))

def process_text(text):
    tokens = word_tokenize(text.lower())
    filtered_tokens = [w for w in tokens if w.isalnum() and w not in stop_words]
    return filtered_tokens

# Separate disaster and non-disaster tweets
disaster_tweets = tweet_train[tweet_train['target'] == 1]['text']
non_disaster_tweets = tweet_train[tweet_train['target'] == 0]['text']

# Process text and compute word frequency
disaster_words = []
for tweet in disaster_tweets:
    disaster_words.extend(process_text(tweet))

non_disaster_words = []
for tweet in non_disaster_tweets:
    non_disaster_words.extend(process_text(tweet))

# Get sentiment scores for words
def filter_sentiment_words(word_list):
    filtered_words = []
    for word in word_list:
        sentiment_score = sia.polarity_scores(word)['compound']
        if abs(sentiment_score) > 0.2:  # Removing neutral words (score ~0)
            filtered_words.append(word)
    return filtered_words

filtered_disaster_words = filter_sentiment_words(disaster_words)
filtered_non_disaster_words = filter_sentiment_words(non_disaster_words)

# Compute word counts
disaster_word_counts = Counter(filtered_disaster_words)
non_disaster_word_counts = Counter(filtered_non_disaster_words)

# Get top 20 words
top_disaster_words = disaster_word_counts.most_common(20)
top_non_disaster_words = non_disaster_word_counts.most_common(20)

# Plot new bar graphs
plt.figure(figsize=(12, 6))

plt.subplot(1, 2, 1)
words, counts = zip(*top_disaster_words)
plt.barh(words, counts, color='red', alpha=0.7)
plt.xlabel('Frequency')
plt.title('Top 20 Meaningful Words in Disaster Tweets')
plt.gca().invert_yaxis()  # Invert to show highest first

plt.subplot(1, 2, 2)
words, counts = zip(*top_non_disaster_words)
plt.barh(words, counts, color='blue', alpha=0.7)
plt.xlabel('Frequency')
plt.title('Top 20 Meaningful Words in Non-Disaster Tweets')
plt.gca().invert_yaxis()  # Invert to show highest first

plt.tight_layout()
plt.show()


In [ ]:
#!pip install contractions
#!pip install nltk
import nltk
# Download necessary NLTK resources
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('wordnet') # Download the wordnet resource

import pandas as pd
import re
import contractions
from nltk import pos_tag, word_tokenize
from nltk.corpus import stopwords, wordnet
from nltk.stem import WordNetLemmatizer
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer
import matplotlib.pyplot as plt

# Initialize NLP tools
stop_words = set(stopwords.words('english') +
                ['rt', 'http', 'https', 'amp', 'com', 'co', 'lol', 'im', 'u'])
lemmatizer = WordNetLemmatizer()

def get_wordnet_pos(treebank_tag):
    """Map POS tag to WordNet tags"""
    if treebank_tag.startswith('J'):
        return wordnet.ADJ
    elif treebank_tag.startswith('V'):
        return wordnet.VERB
    elif treebank_tag.startswith('N'):
        return wordnet.NOUN
    elif treebank_tag.startswith('R'):
        return wordnet.ADV
    else:
        return wordnet.NOUN  # Default to noun

def enhanced_preprocess(text):
    """Advanced text preprocessing with semantic filtering"""
    # Expand contractions
    text = contractions.fix(text)

    # Remove URLs, mentions, hashtags, and special characters
    text = re.sub(r"http\S+|@\w+|#|[\U0001F600-\U0001F6FF]", "", text)
    text = re.sub(r"[^a-zA-Z0-9]", " ", text)

    # Tokenize with POS tagging
    tokens = word_tokenize(text.lower())
    pos_tags = pos_tag(tokens)

    # Filter and lemmatize with POS
    processed = []
    for word, tag in pos_tags:
        if word.isdigit() and len(word) < 4:  # Remove standalone short numbers
            continue
        if word not in stop_words and len(word) > 2:
            lemma = lemmatizer.lemmatize(word, get_wordnet_pos(tag[0]))
            processed.append(lemma)

    return processed
def get_semantic_ngrams(text_series, n=2, top_k=15):
    """Get significant n-grams using TF-IDF weighting"""
    vectorizer = TfidfVectorizer(ngram_range=(n, n),
                               max_features=1000,
                               tokenizer=lambda x: x,
                               preprocessor=lambda x: x)

    tfidf_matrix = vectorizer.fit_transform(text_series)
    feature_names = vectorizer.get_feature_names_out()
    tfidf_scores = tfidf_matrix.sum(axis=0).A1

    return sorted(zip(feature_names, tfidf_scores),
                key=lambda x: x[1], reverse=True)[:top_k]

# Load and preprocess data
tweet_train['processed'] = tweet_train['text'].apply(enhanced_preprocess)

# Split datasets
disaster_processed = tweet_train[tweet_train['target'] == 1]['processed']
non_disaster_processed = tweet_train[tweet_train['target'] == 0]['processed']

# Get TF-IDF weighted n-grams
disaster_bigrams = get_semantic_ngrams(disaster_processed, n=2)
disaster_trigrams = get_semantic_ngrams(disaster_processed, n=3)

non_disaster_bigrams = get_semantic_ngrams(non_disaster_processed, n=2)
non_disaster_trigrams = get_semantic_ngrams(non_disaster_processed, n=3)

# Visualization function with improved styling
def plot_semantic_ngrams(ngrams, title, color):
    labels = [' '.join(gram[0]) for gram in ngrams]
    scores = [gram[1] for gram in ngrams]

    plt.figure(figsize=(12, 8))
    bars = plt.barh(labels, scores, color=color)
    plt.title(f"TF-IDF Weighted {title}", fontsize=14, pad=20)
    plt.xlabel("TF-IDF Score", fontsize=12)
    plt.gca().invert_yaxis()

    # Add value labels
    for bar in bars:
        width = bar.get_width()
        plt.text(width*1.01, bar.get_y() + bar.get_height()/2,
                f'{width:.2f}',
                va='center', ha='left', fontsize=9)

    plt.tight_layout()
    plt.show()

# Generate visualizations
plot_semantic_ngrams(disaster_bigrams, "Disaster Bigrams", '#ff6b6b')
plot_semantic_ngrams(non_disaster_bigrams, "Non-Disaster Bigrams", '#4ecdc4')

plot_semantic_ngrams(disaster_trigrams, "Disaster Trigrams", '#ff6b6b')
plot_semantic_ngrams(non_disaster_trigrams, "Non-Disaster Trigrams", '#4ecdc4')

In [ ]:
import pandas as pd
import re
from nltk.sentiment import SentimentIntensityAnalyzer
import matplotlib.pyplot as plt
import seaborn as sns
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

# Download necessary NLTK data
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('wordnet')

# Initialize VADER sentiment analyzer
sia = SentimentIntensityAnalyzer()

# Add custom lexicon for disaster context
disaster_lexicon = {
    'evacuation': -3.5, 'wildfire': -4.0, 'crash': -4.2,
    'casualties': -4.5, 'fatality': -4.8, 'ablaze': -3.7
}
sia.lexicon.update(disaster_lexicon)

# Define filler words and stopwords
filler_words = set(["um", "uh", "like", "just", "really", "basically", "actually", "literally", "kinda", "sorta"])
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    """Preprocess text: remove noise, stopwords, and filler words; apply lemmatization."""
    text = re.sub(r"http\S+|@\w+|#", "", text)  # Remove URLs, mentions, hashtags
    text = re.sub(r"[^\w\s]", "", text.lower())  # Remove punctuation and lowercase

    tokens = word_tokenize(text)
    cleaned_tokens = [
        lemmatizer.lemmatize(word) for word in tokens
        if word not in stop_words and word not in filler_words
    ]
    return " ".join(cleaned_tokens)

def analyze_sentiment(text):
    """Clean text and analyze sentiment scores."""
    text = clean_text(text)
    return sia.polarity_scores(text)

# Load data (ensure tweet_train is properly defined)
tweet_train['sentiment'] = tweet_train['text'].apply(analyze_sentiment)
tweet_train[['neg', 'neu', 'pos', 'compound']] = tweet_train['sentiment'].apply(pd.Series)

# Split into classes
disaster_sentiment = tweet_train[tweet_train['target'] == 1][['neg', 'neu', 'pos', 'compound']]
non_disaster_sentiment = tweet_train[tweet_train['target'] == 0][['neg', 'neu', 'pos', 'compound']]

from scipy.stats import ttest_ind

# Compare negative sentiment scores
t_stat, p_value = ttest_ind(
    disaster_sentiment['neg'],
    non_disaster_sentiment['neg'],
    alternative='greater'
)

print(f"T-statistic: {t_stat:.2f}, p-value: {p_value:.4f}")

plt.figure(figsize=(10, 6))
sns.kdeplot(disaster_sentiment['neg'], label='Disaster Tweets', fill=True, color='#ff6b6b')
sns.kdeplot(non_disaster_sentiment['neg'], label='Non-Disaster Tweets', fill=True, color='#4ecdc4')
plt.title("Negative Sentiment Distribution Comparison")
plt.xlabel("Negative Sentiment Score")
plt.legend()
plt.show()

import plotly.express as px

# Aggregate sentiment scores
avg_scores = tweet_train.groupby('target')[['neg', 'neu', 'pos']].mean().reset_index()
melted = avg_scores.melt(id_vars='target', var_name='tone', value_name='score')

# Plot radar chart
fig = px.line_polar(
    melted,
    r="score",
    theta="tone",
    color="target",
    line_close=True,
    color_discrete_map={0: '#4ecdc4', 1: '#ff6b6b'},
    title="Emotional Tone Comparison"
)
fig.show()


### 4. RNN Model Construction

In [ ]:
#!pip install tensorflow scikeras scikit-learn

In [ ]:
#!pip install scikit-learn
#!pip install scikeras

In [ ]:
#!pip install scikit-learn==1.2.2 # Downgrade scikit-learn to a compatible version
#!pip install scikeras --upgrade # Make sure scikeras is up-to-date

In [ ]:
#import sklearn
#print(sklearn.__version__)

#### 4.1 Simple RNN Model

In [ ]:
import pandas as pd
import numpy as np
import re
from nltk.corpus import stopwords
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

# Load data
train_file_path = '/path/to/project_folder/tweet_train.csv'
test_file_path = 'path/to/project_folder/tweet_test.csv'

tweet_train = pd.read_csv(train_file_path)
tweet_test = pd.read_csv(test_file_path)

# Text preprocessing
stop_words = set(stopwords.words('english'))

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+', '', text)  # Remove URLs
    text = re.sub(r'[^a-zA-Z\s]', '', text)  # Remove special characters
    text = re.sub(r'\s+', ' ', text).strip()  # Remove extra whitespace
    words = text.split()
    words = [w for w in words if w not in stop_words]
    return ' '.join(words)

tweet_train['cleaned_text'] = tweet_train['text'].apply(clean_text)
tweet_test['cleaned_text'] = tweet_test['text'].apply(clean_text)

# Split training data into train/validation
X_train_full = tweet_train['cleaned_text']
y_train_full = tweet_train['target']
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full,
    test_size=0.2,
    random_state=42,
    stratify=y_train_full
)

# Tokenization and sequencing
max_words = 10000
max_len = 100

tokenizer = Tokenizer(num_words=max_words, oov_token='<OOV>')
tokenizer.fit_on_texts(X_train)

# Convert texts to sequences
X_train_seq = pad_sequences(tokenizer.texts_to_sequences(X_train), maxlen=max_len)
X_val_seq = pad_sequences(tokenizer.texts_to_sequences(X_val), maxlen=max_len)
X_test_seq = pad_sequences(tokenizer.texts_to_sequences(tweet_test['cleaned_text']), maxlen=max_len)

# Model building function
def build_model(embed_dim=128, rnn_units=64, learning_rate=0.001):
    model = Sequential()
    model.add(Embedding(input_dim=max_words, output_dim=embed_dim, input_length=max_len))
    model.add(SimpleRNN(rnn_units, dropout=0.2, recurrent_dropout=0.2))
    model.add(Dense(1, activation='sigmoid'))

    model.compile(
        optimizer=Adam(learning_rate=learning_rate),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model

# Cross-validation
kfold = KFold(n_splits=5, shuffle=True, random_state=42)
fold_no = 1
accuracies = []
f1_scores = []

for train_idx, val_idx in kfold.split(X_train_seq, y_train):
    print(f'\nTraining fold {fold_no}')

    # Data splitting
    X_train_fold, X_val_fold = X_train_seq[train_idx], X_train_seq[val_idx]
    y_train_fold, y_val_fold = y_train.iloc[train_idx], y_train.iloc[val_idx]

    # Model initialization
    model = build_model()

    # Early stopping
    early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

    # Training
    history = model.fit(
        X_train_fold, y_train_fold,
        epochs=20,
        batch_size=64,
        validation_data=(X_val_fold, y_val_fold),
        callbacks=[early_stop],
        verbose=0
    )

    # Evaluation
    y_pred = (model.predict(X_val_fold) > 0.5).astype(int)
    report = classification_report(y_val_fold, y_pred, output_dict=True)

    accuracies.append(report['accuracy'])
    f1_scores.append(report['macro avg']['f1-score'])

    fold_no += 1

print(f'\nCross-Validation Results:')
print(f'Average Accuracy: {np.mean(accuracies):.4f} (±{np.std(accuracies):.4f})')
print(f'Average F1-Score: {np.mean(f1_scores):.4f} (±{np.std(f1_scores):.4f})')

# Final training on full training data
final_model = build_model()
history = final_model.fit(
    X_train_seq, y_train,
    epochs=20,
    batch_size=64,
    validation_data=(X_val_seq, y_val),
    callbacks=[early_stop],
    verbose=1
)

# Generate predictions for test set
test_predictions = (final_model.predict(X_test_seq) > 0.5).astype(int).flatten()

# Create submission file
submission = pd.DataFrame({
    'id': tweet_test['id'],
    'target': test_predictions
})

file_path = 'path/to/project_folder/Module 1/'

submission.to_csv(file_path + 'disaster_tweet_predictions.csv', index=False)
print("\nTest predictions saved to disaster_tweet_predictions.csv")

# Validation set evaluation
print('\nValidation Set Evaluation:')
y_val_pred = (final_model.predict(X_val_seq) > 0.5).astype(int)
print(classification_report(y_val, y_val_pred))
print('Confusion Matrix:')
print(confusion_matrix(y_val, y_val_pred))

#### 4.2 RNN Model: lementization and class weights added

In [ ]:
#!pip install joblib

In [ ]:
import pandas as pd
import numpy as np
import re
import nltk
import tensorflow as tf
import keras_tuner as kt
from nltk.corpus import stopwords
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from joblib import Parallel, delayed

# Configuration
nltk.download(['stopwords', 'wordnet'])
tf.keras.mixed_precision.set_global_policy('mixed_float16')

# Preprocessing components
stop_words = set(stopwords.words('english'))
lemmatizer = nltk.WordNetLemmatizer()
token_pattern = re.compile(r'\b[a-z]{3,15}\b')

def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'[^a-z\s]', '', text)
    words = token_pattern.findall(text)
    return ' '.join([lemmatizer.lemmatize(w) for w in words if w not in stop_words])

def parallel_preprocess(texts):
    return Parallel(n_jobs=-1)(delayed(preprocess_text)(t) for t in texts)

# Load and preprocess data
train_file_path = '/path/to/project_folder/tweet_train.csv'
test_file_path = '/path/to/project_folder/tweet_test.csv'

# Preprocess data
train_df['cleaned'] = parallel_preprocess(train_df['text'])
test_df['cleaned'] = parallel_preprocess(test_df['text'])

# Split training data into train/validation
train_data, val_data = train_test_split(
    train_df,
    test_size=0.2,
    stratify=train_df['target'],
    random_state=42
)

# Tokenization setup
max_words = 8000
max_len = 50
tokenizer = Tokenizer(num_words=max_words, oov_token='<OOV>', filters='', lower=False)
tokenizer.fit_on_texts(train_data['cleaned'])

# Dataset creation
def create_dataset(texts, labels=None, batch_size=512):
    seq = tokenizer.texts_to_sequences(texts)
    padded = pad_sequences(seq, maxlen=max_len, dtype='int32', padding='post')
    if labels is not None:
        return tf.data.Dataset.from_tensor_slices((padded, labels))
    return tf.data.Dataset.from_tensor_slices(padded)

# Create TensorFlow datasets (using a smaller subset of data)
train_ds = create_dataset(train_data['cleaned'][:len(train_data)//10], train_data['target'][:len(train_data)//10]
                          ).cache().shuffle(10000).batch(64).prefetch(tf.data.AUTOTUNE)
val_ds = create_dataset(val_data['cleaned'][:len(val_data)//10], val_data['target'][:len(val_data)//10]
                        ).cache().batch(64).prefetch(tf.data.AUTOTUNE)

# Hyperparameter tuning setup (reduced max_epochs, iterations, and hyperparameter search space)
def model_builder(hp):
    model = tf.keras.Sequential([
        tf.keras.layers.Embedding(
            input_dim=max_words + 1,
            output_dim=hp.Int('embedding_dim', 64, 96, step=32),  # Reduced range
            mask_zero=True),
        tf.keras.layers.GRU(
            hp.Int('gru_units', 16, 32, step=16),  # Reduced units
            dropout=hp.Float('dropout', 0.2, 0.3, step=0.05),  # Reduced range
            recurrent_dropout=0.2,
            kernel_regularizer=tf.keras.regularizers.l2(0.001)),
        tf.keras.layers.Dense(1, activation='sigmoid', dtype='float32')
    ])
    model.compile(
        optimizer=tf.keras.optimizers.Adam(hp.Choice('learning_rate', [1e-3])),
        loss='binary_crossentropy',
        metrics=['accuracy', tf.keras.metrics.Precision(name='precision'),
                 tf.keras.metrics.Recall(name='recall')]
    )
    return model

# Class weights
class_weight = {
    0: len(train_data) / (2 * (len(train_data) - train_data.target.sum())),
    1: len(train_data) / (2 * train_data.target.sum())
}

# Hyperparameter search (reduced max_epochs and iterations)
tuner = kt.Hyperband(
    model_builder,
    objective=kt.Objective('val_recall', direction='max'),
    max_epochs=3,  # Reduced from 15
    factor=3,
    hyperband_iterations=1,  # Reduced from 2
    overwrite=True
)

tuner.search(
    train_ds,
    validation_data=val_ds,
    epochs=3,  # Reduced epochs
    class_weight=class_weight,
    callbacks=[tf.keras.callbacks.EarlyStopping(patience=1, restore_best_weights=True)]  # Early stopping with reduced patience
)

# Cross-validation (reduced to 1 fold or removed)
best_hps = tuner.get_best_hyperparameters()[0]
skf = StratifiedKFold(n_splits=2)

fold_metrics = []
for train_idx, val_idx in skf.split(train_df['cleaned'], train_df['target']):
    train_fold = train_df.iloc[train_idx]
    val_fold = train_df.iloc[val_idx]

    # Re-fit tokenizer for each fold to prevent leakage
    fold_tokenizer = Tokenizer(num_words=max_words, oov_token='<OOV>', filters='', lower=False)
    fold_tokenizer.fit_on_texts(train_fold['cleaned'])

    # Create fold datasets
    fold_train_ds = create_dataset(train_fold['cleaned'], train_fold['target']
                                  ).cache().shuffle(10000).batch(64).prefetch(tf.data.AUTOTUNE)
    fold_val_ds = create_dataset(val_fold['cleaned'], val_fold['target']
                                ).cache().batch(64).prefetch(tf.data.AUTOTUNE)

    # Build fresh model with best params
    model = tuner.hypermodel.build(best_hps)

    # Train model with early stopping
    history = model.fit(
        fold_train_ds,
        validation_data=fold_val_ds,
        epochs=3,  # Reduced epochs
        class_weight=class_weight,
        verbose=0,  # Reduced verbosity
        callbacks=[tf.keras.callbacks.EarlyStopping(patience=1)]
    )

    val_preds = (model.predict(fold_val_ds) > 0.5).astype(int).flatten()
    fold_metrics.append(f1_score(val_fold['target'], val_preds, average='macro'))

print(f"\nCross-Validation Macro F1: {np.mean(fold_metrics):.4f} (±{np.std(fold_metrics):.4f})")

# Final model training on full data (with reduced epochs and batch size)
full_train_ds = create_dataset(train_df['cleaned'][:len(train_df)//10], train_df['target'][:len(train_df)//10]
                       ).cache().shuffle(10000).batch(64).prefetch(tf.data.AUTOTUNE)

final_model = tuner.hypermodel.build(best_hps)
final_model.fit(
    full_train_ds,
    epochs=3,  # Reduced epochs
    class_weight=class_weight,
    callbacks=[tf.keras.callbacks.EarlyStopping(patience=1)]
)

# Generate test predictions (no target access)
test_ds = create_dataset(test_df['cleaned']).batch(64)  # Using smaller batch
test_preds = (final_model.predict(test_ds) > 0.5).astype(int).flatten()

# Generate validation predictions from original holdout set - **CHANGE HERE**
val_ds = create_dataset(val_data['cleaned']).batch(64)  # Use the full validation set
val_preds = (final_model.predict(val_ds) > 0.5).astype(int).flatten()

# Enhanced metrics reporting
print("\n=== Comprehensive Validation Metrics ===")
print(classification_report(val_data['target'], val_preds, target_names=["Non-Disaster", "Disaster"]))
print("Confusion Matrix:")
print(confusion_matrix(val_data['target'], val_preds))

# Detailed metrics extraction
report = classification_report(val_data['target'], val_preds, output_dict=True)
macro_f1 = f1_score(val_data['target'], val_preds, average='macro')

print("\n=== Strategic Performance Summary ===")
print(f"Overall Accuracy: {report['accuracy']:.4f}")
print(f"Macro Average F1: {macro_f1:.4f}")
print(f"Disaster Class Recall (Sensitivity): {report['1']['recall']:.4f}")
print(f"Non-Disaster Class Precision (Specificity): {report['0']['precision']:.4f}")
print(f"F1 Scores - Non-Disaster: {report['0']['f1-score']:.4f} | Disaster: {report['1']['f1-score']:.4f}")
print(f"Support Samples - Non-Disaster: {report['0']['support']} | Disaster: {report['1']['support']}")

# Create submission file
submission = pd.DataFrame({
    'id': test_df['id'],
    'target': test_preds
})

file_path = 'path/to/project_folder/'
submission.to_csv(file_path + 'disaster_tweet_predictions_updates_1.csv', index=False)


#### 4.3 RNN Model: lemmatization with an attempt to optimize model performance vs optimal training time

In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import StratifiedKFold
import keras_tuner as kt
from joblib import Parallel, delayed
import re
import nltk
from nltk.corpus import stopwords
from tensorflow.keras.layers import Embedding, GRU, Dense
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split

# Download NLTK stopwords
nltk.download(['stopwords', 'wordnet'])

# Configuration and Preprocessing Components
stop_words = set(stopwords.words('english'))
lemmatizer = nltk.WordNetLemmatizer()
token_pattern = re.compile(r'\b[a-z]{3,15}\b')

def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'[^a-z\s]', '', text)
    words = token_pattern.findall(text)
    return ' '.join([lemmatizer.lemmatize(w) for w in words if w not in stop_words])

def parallel_preprocess(texts):
    return Parallel(n_jobs=-1)(delayed(preprocess_text)(t) for t in texts)

# Load Data
train_file_path = 'path/to/project_folder/tweet_train.csv'
test_file_path = 'path/to/project_folder/tweet_test.csv'
train_df = pd.read_csv(train_file_path)
test_df = pd.read_csv(test_file_path)

# Preprocess Data
train_df['cleaned'] = parallel_preprocess(train_df['text'])
test_df['cleaned'] = parallel_preprocess(test_df['text'])

# Split Training Data into Train/Validation
train_data, val_data = train_test_split(
    train_df,
    test_size=0.2,
    stratify=train_df['target'],
    random_state=42
)

# Tokenizer Setup
max_words = 8000
max_len = 50
tokenizer = Tokenizer(num_words=max_words, oov_token='<OOV>', filters='', lower=False)
tokenizer.fit_on_texts(train_data['cleaned'])

# Dataset Creation
def create_dataset(texts, labels=None, batch_size=512):
    seq = tokenizer.texts_to_sequences(texts)
    padded = pad_sequences(seq, maxlen=max_len, dtype='int32', padding='post')
    if labels is not None:
        return tf.data.Dataset.from_tensor_slices((padded, labels))
    return tf.data.Dataset.from_tensor_slices(padded)

# Hyperparameter Search Setup
def model_builder(hp):
    model = tf.keras.Sequential([
        Embedding(
            input_dim=max_words + 1,
            output_dim=hp.Int('embedding_dim', 64, 128, step=32),
            mask_zero=True
        ),
        GRU(
            hp.Int('gru_units', 32, 128, step=32),
            dropout=hp.Float('dropout', 0.1, 0.5, step=0.1),
            recurrent_dropout=0.2,
            kernel_regularizer=tf.keras.regularizers.l2(0.001)
        ),
        Dense(1, activation='sigmoid', dtype='float32')
    ])
    model.compile(
        optimizer=tf.keras.optimizers.Adam(hp.Choice('learning_rate', [1e-3, 5e-4])),
        loss='binary_crossentropy',
        metrics=['accuracy', tf.keras.metrics.Precision(name='precision'),
                 tf.keras.metrics.Recall(name='recall')]
    )
    return model

# Hyperparameter Tuning
tuner = kt.Hyperband(
    model_builder,
    objective=kt.Objective('val_recall', direction='max'),
    max_epochs=3,  # Reduced epochs for faster tuning
    factor=3,
    hyperband_iterations=1,  # Fewer iterations for faster results
    overwrite=True
)

# Create Datasets for Training and Validation
train_ds = create_dataset(train_data['cleaned'], train_data['target']).cache().shuffle(10000).batch(512).prefetch(tf.data.AUTOTUNE)
val_ds = create_dataset(val_data['cleaned'], val_data['target']).cache().batch(512).prefetch(tf.data.AUTOTUNE)

# Train Hyperparameter Tuner
tuner.search(train_ds, validation_data=val_ds, epochs=3, class_weight={0: 1.0, 1: 1.0})

# Get Best Hyperparameters
best_hps = tuner.get_best_hyperparameters()[0]

# Final Model Training with Best Hyperparameters
final_model = tuner.hypermodel.build(best_hps)
final_model.fit(
    train_ds,
    epochs=3,  # Limit to 3 epochs for faster training
    batch_size=64,
    class_weight={0: 1.0, 1: 1.0},
    callbacks=[EarlyStopping(patience=1, restore_best_weights=True)]
)

# Generate Validation Predictions
val_preds = (final_model.predict(val_ds) > 0.5).astype(int).flatten()

# Comprehensive Validation Metrics
print("\n=== Comprehensive Validation Metrics ===")
print(classification_report(val_data['target'], val_preds, target_names=["Non-Disaster", "Disaster"]))
print("Confusion Matrix:")
print(confusion_matrix(val_data['target'], val_preds))

# Detailed Metrics Extraction
report = classification_report(val_data['target'], val_preds, output_dict=True)
macro_f1 = f1_score(val_data['target'], val_preds, average='macro')

print("\n=== Strategic Performance Summary ===")
print(f"Overall Accuracy: {report['accuracy']:.4f}")
print(f"Macro Average F1: {macro_f1:.4f}")
print(f"Disaster Class Recall (Sensitivity): {report['1']['recall']:.4f}")
print(f"Non-Disaster Class Precision (Specificity): {report['0']['precision']:.4f}")
print(f"F1 Scores - Non-Disaster: {report['0']['f1-score']:.4f} | Disaster: {report['1']['f1-score']:.4f}")
print(f"Support Samples - Non-Disaster: {report['0']['support']} | Disaster: {report['1']['support']}")

# Generate Test Predictions
test_ds = create_dataset(test_df['cleaned']).batch(512)
test_preds = (final_model.predict(test_ds) > 0.5).astype(int).flatten()

# Create Submission File
submission = pd.DataFrame({
    'id': test_df['id'],
    'target': test_preds
})

# Define file path for saving the submission
file_path = 'path/to/project_folder'
submission_file = file_path + 'disaster_tweet_predictions_updates_2.csv'

# Export Submission File
submission.to_csv(submission_file, index=False)

print(f"\nSubmission file saved to {submission_file}")
